In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def token_report(text):
    toks = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(toks)
    print(f"Texto: {text}")
    print(f"Tokens ({len(toks)}): {toks}")
    print(f"IDs (primeros 12): {ids[:12]}")
    print("-" * 60)

## Parte 1 - Tokenización
Los LLM y transformers necesitan realizar un proceso de "tokenización". Dividen el texto en subpalabras/tokens, el contexto y el peso de cada token se calculan después en el modelo.

In [ ]:
demo_texts = [
    "Programación",
    "Programacion",
    "The cake is a lie",
    "Hay que mover el cacharro",
    "Dónde caemos gente?"
]
for t in demo_texts:
    token_report(t)

Mientras más tokens, más coste y más longitud de secuencia. Cuesta más saber completamente el contexto.
Muchas veces el tokenizer no necesita saber las palabras completas, con conocer la raíz le sirve para saber el contexto:
- habl - o
- habl - aré
- habl - ador

Esto es muy útil en idiomas con palabras extremadamente largas, ya que permite conocer cada una de las raices de las palabras que lo componen

In [ ]:
demo_texts = [
    "Donaudampfschifffahrtsgesellschaftskapitän",
    "Muvaffakiyetsezlestiricilestiriveremeyebileceklerimizdenmissinizcesine",
    "Pneumonoultramicroscopicsilicovolcanoconiosis"
]
for t in demo_texts:
    token_report(t)

## Ejercicio 1
Ver cómo se dividen las siguientes palabras en subpartes frecuentes:
- Electroencefalografista
- Otorrinolaringólogo
- Anticonstitucionalmente

In [ ]:
tarea1_texts = [
    "Electroencefalografista",
    "Otorrinolaringólogo",
    "Anticonstitucionalmente",
]
for t in tarea1_texts:
    token_report(t)

## Ejercicio 2
Algunos tokenizers pueden funcionar en varios idiomas, pero otros pueden tener fallos al ver las diferentes relaciones y calcular la atención.

Eficiencia en diferentes idiomas, ¿cuál es más eficiente?

In [ ]:
tarea2_texts = {
    "es": "Necesito dinero del banco",
    "en": "I need money from the bank",
    "tr": "Bankadan para cekmem gerekiyor",
    "de": "Ich brauche Geld von der Bank",
}

results = []
for lang, text in tarea2_texts.items():
    toks = tokenizer.tokenize(text)
    results.append((lang, text, len(toks), toks))

for lang, text, n, toks in results:
    print(f"{lang} | {n} tokens | {text}")
    print(toks)
    print("-" * 60)

## Ejercicio 3
Análisis de tokenizers, comparar 3 diferentes tokenizers con el mismo dataset.
 - Crear el tokenizer de cada uno de los modelos
 - Tokenizar la cadena de texto entrante
 - Convertir los tokens a ID


In [ ]:
tarea3_tests = [
    "sudo rm -rf / --no-preserve-root",  #Comando de linux
    "https://store.steampowered.com/app/753640/Outer_Wilds/",  #URL
    "なぜそれを翻訳するのですか 💀💀💀",  #Emojis y texto
    "#skillissue #estolocambiatodo #estonocambianada",  #Hashtags
    "AAAAAAA BBBBBB CCCCCC",  #Palabras largas repetidas
    "SELECT * FROM Users WHERE UserId = 105 OR 1=1;", #SQL
    "function test() { return x * 2; }", #Código
    "𓀀𓀁𓀂𓀃𓀄" #Caracteres extraños
]

bert_model_name = "bert-base-multilingual-cased"
tokenizer_bert = None #TODO: crear tokenizer de BERT

xlm_roberta_model_name = "xlm-roberta-base"
tokenizer_xlm = None #TODO: crear tokenizer de xlm

gpt2_model_name = "gpt2"
tokenizer_gpt2 = None #TODO: crear tokenizer de gpt2

models = [tokenizer_bert, tokenizer_xlm, tokenizer_gpt2]

def token_report_model(text,tokenizer):
    toks = None #TODO: tokenizar el texto con el tokenizer dado
    ids = None #TODO: convertir los tokens a IDs con el tokenizer dado

    print(f"Texto: {text}")
    print(f"Tokens ({len(toks)}): {toks}")
    print(f"IDs (primeros 12): {ids[:12]}")
    print("-" * 60)

for m in models:
    for t in tarea3_tests:
        token_report_model(t, m)

## Ejercicio 4 - Embedding
Además de tokenizar, estos modelos preparan las palabras para el siguiente paso del transformer. A este paso se le llama embedding.

- Construye dos frases distintas que el modelo considere muy parecidas
- Construye dos frases que compartan la palabra banco pero que salgan poco parecidas
- Intenta engañar al modelo con faltas de ortografía o palabras que no se espere

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

task4_texts = [
    #TODO
]

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    #promedio simple de embeddings de todos los tokens
    emb = outputs.last_hidden_state.mean(dim=1)
    return emb

embeddings = [sentence_embedding(s) for s in task4_texts]

for i in range(len(task4_texts)):
    for j in range(i + 1, len(task4_texts)):
        sim = F.cosine_similarity(embeddings[i], embeddings[j]).item()
        print(f"{i}-{j}: {sim:.4f} | {task4_texts[i]}  <->  {task4_texts[j]}")